In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Augment a P-Dependent Noise-Robustness results workbook with analysis sheets.
================================================================================

Companion to augment_pdep_report.py, adapted for the noise-injection study
produced by noise_pdep_test_seed62.py -- a raw results sheet with columns:

    Fold | System | Sigma | Rule | p | EER | Rank1

(EER and Rank1 stored as fractions in [0, 1], one row per fold x system x
noise level x rule x p combination; p swept at {0.0, 0.5, 1.0}, sigma at
{0.05, 0.10, 0.15, 0.20}, on the 5-fold TEST sets, seed=62).

This adds the "usual" set of analysis sheets, adapted to include the extra
Sigma (noise standard deviation) dimension that the TRAIN-sweep workbook
does not have. Two sheets are reoriented around Sigma instead of p, since
noise level -- not p -- is the primary variable under study here; this
substitution is explained explicitly in the Methodology sheet.

Sheets added (or refreshed if already present):
  1. Methodology                  - detailed description of data and methods
  2. Best_By_EER                  - per (System, Sigma) winner on mean EER
  3. Best_by_R1FAR1                - per (System, Sigma) winner on mean Rank-1@FAR1%
  4. EER_Sensitivity_Analysis      - mean EER per (System,Sigma) x Rule, avg over p
  5. R1FAR1_Sensitivity_Analysis   - mean Rank-1 per (System,Sigma) x Rule, avg over p
  6. P_Sensitivity_Analysis        - EER-optimal p per Rule x System x Sigma (long form)
  7. BestEER_vs_Sigma_AllRules     - best achievable EER vs sigma, min over all rules & p
  8. EER_vs_Sigma_WinningConfig    - EER vs sigma, using each system's single overall
                                     winning (rule, p) held fixed across noise levels

USAGE
-----
    python augment_noise_report.py noise_pdep_test_seed62_raw.xlsx
================================================================================
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from scipy import stats
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.worksheet import Worksheet


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

RAW_XLSX_FILE = "./Results/noise_pdep_test_seed62_raw.xlsx"
RESULTS_SHEET_NAME = "Raw"          # first/only sheet in the source workbook
RESULTS_HEADER_ROW = 0              # plain export: header on the very first row
EER_COL = "EER"                     # source column, stored as a FRACTION [0,1]
RANK1_COL = "Rank1"                 # source column, stored as a FRACTION [0,1]

THEME_COLOR = "B45309"              # orange, matching the earlier noise-study workbook
FONT_NAME = "Arial"
CONFIDENCE_LEVEL = 0.95


# ==============================================================================
#  STEP 0 -- LOAD THE SOURCE DATA
# ==============================================================================

def load_results(filepath: Path) -> pd.DataFrame:
    """
    Read the raw (Fold, System, Sigma, Rule, p, EER, Rank1) table and rescale
    EER/Rank1 from fractions to percentages for all downstream reporting.
    """
    xls = pd.ExcelFile(filepath)
    sheet_name = RESULTS_SHEET_NAME if RESULTS_SHEET_NAME in xls.sheet_names else xls.sheet_names[0]
    frame = pd.read_excel(filepath, sheet_name=sheet_name, header=RESULTS_HEADER_ROW)

    required = {"Fold", "System", "Sigma", "Rule", "p", EER_COL, RANK1_COL}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(
            f"Source sheet is missing expected columns: {sorted(missing)}. "
            f"Found columns: {list(frame.columns)}"
        )

    frame = frame.copy()
    frame["EER (%)"] = frame[EER_COL] * 100.0
    frame["Rank-1 @ FAR=1% (%)"] = frame[RANK1_COL] * 100.0
    return frame


def system_order(frame: pd.DataFrame) -> List[str]:
    canonical = [f"S{i}" for i in range(1, 9)]
    present = set(frame["System"])
    return [s for s in canonical if s in present]


def rule_order(frame: pd.DataFrame) -> List[str]:
    canonical = ["Zimmermann1", "Zimmermann2", "Luhandjula", "Werners1", "Werners2",
                "Sales1", "Sales2", "Mizumoto1", "Mizumoto2", "R5p"]
    present = set(frame["Rule"])
    ordered = [r for r in canonical if r in present]
    ordered += sorted(present - set(ordered))
    return ordered


def sigma_order(frame: pd.DataFrame) -> List[float]:
    return sorted(frame["Sigma"].unique().tolist())


def p_order(frame: pd.DataFrame) -> List[float]:
    return sorted(frame["p"].unique().tolist())


# ==============================================================================
#  STEP 1 -- STATISTICS HELPERS
# ==============================================================================

def ci95_half_width(values: pd.Series) -> float:
    n = values.count()
    if n < 2:
        return 0.0
    alpha = 1.0 - CONFIDENCE_LEVEL
    t_crit = stats.t.ppf(1.0 - alpha / 2.0, df=n - 1)
    return float(t_crit * values.std(ddof=1) / np.sqrt(n))


# ==============================================================================
#  STEP 2 -- AGGREGATE ACROSS FOLDS FIRST (mean-first, as the corrected protocol requires)
# ==============================================================================

def aggregate_per_config(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Average EER (%) and Rank-1 (%) across the 5 folds for every
    (System, Sigma, Rule, p) combination. Every summary below is built from
    this table, so every reported winner is one fixed configuration.
    """
    grouped = frame.groupby(["System", "Sigma", "Rule", "p"], as_index=False).agg(
        EER_mean=("EER (%)", "mean"),
        EER_std=("EER (%)", "std"),
        Rank1_mean=("Rank-1 @ FAR=1% (%)", "mean"),
        Rank1_std=("Rank-1 @ FAR=1% (%)", "std"),
        N_folds=("Fold", "count"),
    )
    grouped["EER_std"] = grouped["EER_std"].fillna(0.0)
    grouped["Rank1_std"] = grouped["Rank1_std"].fillna(0.0)
    return grouped


# ==============================================================================
#  STEP 3 -- BUILD EACH TABLE
# ==============================================================================

def build_best_by_eer(per_config: pd.DataFrame, systems: List[str],
                      sigmas: List[float]) -> pd.DataFrame:
    """Sheet 2: for each (System, Sigma), the (rule, p) with lowest mean EER."""
    rows = []
    for system in systems:
        for sigma in sigmas:
            subset = per_config[(per_config.System == system) & (per_config.Sigma == sigma)]
            if subset.empty:
                continue
            best = subset.loc[subset.EER_mean.idxmin()]
            rows.append({
                "System": system, "Sigma": sigma, "Rule": best.Rule, "p": best.p,
                "EER (%)": round(best.EER_mean, 4),
                "Rank-1 (%)": round(best.Rank1_mean, 4),
                "Rank-1 @ FAR=1%": round(best.Rank1_mean, 4),
            })
    return pd.DataFrame(rows)


def build_best_by_rank1(per_config: pd.DataFrame, systems: List[str],
                        sigmas: List[float]) -> pd.DataFrame:
    """Sheet 3: for each (System, Sigma), the (rule, p) with highest mean Rank-1."""
    rows = []
    for system in systems:
        for sigma in sigmas:
            subset = per_config[(per_config.System == system) & (per_config.Sigma == sigma)]
            if subset.empty:
                continue
            best = subset.loc[subset.Rank1_mean.idxmax()]
            rows.append({
                "System": system, "Sigma": sigma, "Rule": best.Rule, "p": best.p,
                "EER (%)": round(best.EER_mean, 4),
                "Rank-1 (%)": round(best.Rank1_mean, 4),
                "Rank-1 @ FAR=1%": round(best.Rank1_mean, 4),
            })
    return pd.DataFrame(rows)


def build_sensitivity(per_config: pd.DataFrame, systems: List[str], rules: List[str],
                      sigmas: List[float], value_col: str) -> pd.DataFrame:
    """
    Sheets 4 & 5: mean value per (System, Sigma, Rule), averaged over all p.
    Rows are (System, Sigma) pairs; columns are rules.
    """
    pivot = (
        per_config.groupby(["System", "Sigma", "Rule"])[value_col].mean()
        .unstack("Rule").reindex(columns=rules)
    )
    pivot = pivot.reindex(
        pd.MultiIndex.from_product([systems, sigmas], names=["System", "Sigma"])
    )
    pivot = pivot.reset_index()
    return pivot


def build_p_sensitivity(per_config: pd.DataFrame, systems: List[str], rules: List[str],
                        sigmas: List[float]) -> pd.DataFrame:
    """
    Sheet 6: for every (Rule, System, Sigma), the p minimizing mean EER, and
    the EER achieved there. Kept as a long table (rather than a wide pivot)
    because adding Sigma as a third dimension makes a single wide table
    unreadable; this form stays exact and filterable in Excel.
    """
    idx = per_config.groupby(["System", "Sigma", "Rule"])["EER_mean"].idxmin()
    best = per_config.loc[idx, ["System", "Sigma", "Rule", "p", "EER_mean"]].copy()
    best = best.rename(columns={"p": "Optimal_p", "EER_mean": "EER (%) at Optimal_p"})
    best["EER (%) at Optimal_p"] = best["EER (%) at Optimal_p"].round(4)
    best["System"] = pd.Categorical(best["System"], systems, ordered=True)
    best["Rule"] = pd.Categorical(best["Rule"], rules, ordered=True)
    return best.sort_values(["Rule", "System", "Sigma"]).reset_index(drop=True)


def build_best_eer_vs_sigma(per_config: pd.DataFrame, systems: List[str],
                            sigmas: List[float]) -> pd.DataFrame:
    """
    Sheet 7: for each sigma, the BEST achievable mean EER -- minimized over
    all rules AND all p -- per system. The noise-robustness "ceiling" curve.
    """
    pivot = (
        per_config.groupby(["System", "Sigma"])["EER_mean"].min()
        .unstack("System").reindex(index=sigmas, columns=systems)
    )
    pivot.insert(0, "Sigma", pivot.index)
    return pivot.reset_index(drop=True)


def build_eer_vs_sigma_winning_config(per_config: pd.DataFrame, systems: List[str],
                                      sigmas: List[float]) -> pd.DataFrame:
    """
    Sheet 8: pick ONE overall winning (rule, p) per system -- the config with
    the lowest mean EER averaged ACROSS ALL SIGMA LEVELS -- then show how that
    single fixed configuration's EER changes as sigma increases. Unlike sheet
    7, the configuration never changes here: this shows how a config chosen
    without knowing the noise level in advance actually holds up under it.
    """
    overall_mean = (
        per_config.groupby(["System", "Rule", "p"])["EER_mean"].mean().reset_index()
    )
    winners = {}
    for system in systems:
        subset = overall_mean[overall_mean.System == system]
        if subset.empty:
            continue
        best = subset.loc[subset.EER_mean.idxmin()]
        winners[system] = (best.Rule, best.p)

    columns = {}
    for system in systems:
        if system not in winners:
            continue
        rule, p = winners[system]
        subset = per_config[
            (per_config.System == system) & (per_config.Rule == rule) & (per_config.p == p)
        ]
        series = subset.set_index("Sigma")["EER_mean"].reindex(sigmas)
        columns[system] = series
    result = pd.DataFrame(columns)
    result.insert(0, "Sigma", result.index)
    result = result.reset_index(drop=True)

    winner_row = {"Sigma": "Rule (p)"}
    for system in systems:
        if system in winners:
            rule, p = winners[system]
            winner_row[system] = f"{rule} (p={p})"
    result = pd.concat([pd.DataFrame([winner_row]), result], ignore_index=True)
    return result


# ==============================================================================
#  STEP 4 -- METHODOLOGY TEXT
# ==============================================================================

def methodology_lines(frame: pd.DataFrame, systems: List[str], rules: List[str],
                      sigmas: List[float], p_values: List[float]) -> List[tuple]:
    n_folds = int(frame["Fold"].nunique())

    noisy_modality = {
        "S1": "Face_G", "S2": "Finger_LI", "S3": "Finger_RI", "S4": "Finger_LI",
        "S5": "Finger_RI", "S6": "Finger_RI", "S7": "Face_C (3rd/stage-2 modality)",
        "S8": "Face_G (3rd/stage-2 modality)",
    }
    noisy_text = "; ".join(f"{s}: {noisy_modality[s]}" for s in systems if s in noisy_modality)

    return [
        ("title", "Methodology"),
        ("note",  "How every derived sheet in this workbook was computed from the Raw sheet."),
        ("blank", ""),
        ("head",  "SOURCE DATA AND NOISE INJECTION"),
        ("body",  f"The 'Raw' sheet holds {len(frame)} rows: {n_folds} cross-validation TEST folds "
                  f"x {len(systems)} fusion systems x {len(sigmas)} noise levels x {len(rules)} "
                  f"parametrized rules x {len(p_values)} values of p (p \u2208 "
                  f"{{{', '.join(str(p) for p in p_values)}}}). Normalization (min-max) was fit on "
                  "each fold's TRAIN submatrix only and applied to that fold's TEST submatrix -- the "
                  "same protocol as every other TEST evaluation in this study."),
        ("body",  "For every system, ONE modality was kept clean and the other was corrupted with "
                  f"zero-mean Gaussian noise N(0, sigma), sigma \u2208 "
                  f"{{{', '.join(str(s) for s in sigmas)}}}, added to the normalized TEST scores and "
                  "clipped back to [0, 1]. The corrupted modality per system: " + noisy_text + "."),
        ("body",  "For the two-stage cascaded systems (S7, S8), BOTH fingerprint modalities "
                  "(Finger_LI, Finger_RI) stayed clean at the first fusion stage; noise was injected "
                  "only into the face modality added at the second stage, since that is the '3rd' "
                  "modality referred to in the study brief."),
        ("blank", ""),
        ("head",  "AGGREGATION ORDER (why this matters)"),
        ("body",  "As in every other workbook in this study, results are averaged ACROSS THE 5 FOLDS "
                  "FIRST, independently for every (System, Sigma, Rule, p). Only after that averaging "
                  "does any 'best' selection happen. This keeps every reported winner a single, fixed, "
                  "deployable configuration rather than an optimistic blend of different per-fold choices."),
        ("blank", ""),
        ("head",  "WHY TWO SHEETS ARE BUILT AROUND SIGMA INSTEAD OF p"),
        ("body",  "The TRAIN-sweep workbook's sheets 7 and 8 ('best EER vs p', 'EER vs p for the "
                  "winning rule') existed because p was the variable under study there. Here, NOISE "
                  "LEVEL is the primary variable of interest, so sheets 7 and 8 are built around Sigma "
                  "instead: 'best achievable EER vs sigma' (the ceiling, re-optimizing rule and p at "
                  "every noise level) and 'EER vs sigma for ONE fixed winning configuration' (the "
                  "realistic case of a system that was tuned once, then exposed to varying noise)."),
        ("blank", ""),
        ("head",  "SHEET-BY-SHEET DEFINITIONS"),
        ("body",  "Best_By_EER / Best_by_R1FAR1: per (System, Sigma), the (rule, p) with the lowest "
                  "mean EER / highest mean Rank-1, independently re-optimized at every noise level."),
        ("body",  "EER_Sensitivity_Analysis / R1FAR1_Sensitivity_Analysis: mean value per "
                  "(System, Sigma, Rule), averaged over all 3 values of p -- shows how a rule performs "
                  "'on average' at each noise level, independent of how well p happens to be tuned."),
        ("body",  "P_Sensitivity_Analysis: for every (Rule, System, Sigma), the p minimizing mean EER "
                  "and the EER achieved there, as a long table (a full wide pivot across System x "
                  "Sigma would be unreadable with 3 dimensions)."),
        ("body",  "BestEER_vs_Sigma_AllRules: the noise-robustness ceiling -- at each sigma, the best "
                  "mean EER achievable by ANY rule and p, per system."),
        ("body",  "EER_vs_Sigma_WinningConfig: ONE (rule, p) per system, selected as the configuration "
                  "with the lowest EER averaged ACROSS ALL SIGMA LEVELS (i.e. chosen without knowing "
                  "the noise level in advance), then evaluated at each sigma individually. This is the "
                  "realistic degradation curve for a system tuned once and deployed under varying noise."),
    ]


# ==============================================================================
#  STEP 5 -- EXCEL WRITING HELPERS
# ==============================================================================

def _styles():
    thin = Side(style="thin", color="D9D9D9")
    return {
        "header_fill": PatternFill("solid", fgColor=THEME_COLOR),
        "header_font": Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10),
        "title_font":  Font(name=FONT_NAME, bold=True, size=13, color=THEME_COLOR),
        "note_font":   Font(name=FONT_NAME, italic=True, size=9, color="555555"),
        "cell_font":   Font(name=FONT_NAME, size=10),
        "border":      Border(left=thin, right=thin, top=thin, bottom=thin),
    }


def _autofit(worksheet: Worksheet, n_columns: int, header_row: int, n_rows: int,
            minimum: int = 9, maximum: int = 32) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


def _write_table(worksheet: Worksheet, frame: pd.DataFrame, styles: dict,
                 title: str, subtitle: str = "", percent_cols: tuple = ()) -> int:
    worksheet.cell(row=1, column=1, value=title).font = styles["title_font"]
    row = 2
    if subtitle:
        worksheet.cell(row=row, column=1, value=subtitle).font = styles["note_font"]
        row += 1
    row += 1
    header_row = row

    for col_idx, column in enumerate(frame.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=str(column))
        cell.fill = styles["header_fill"]
        cell.font = styles["header_font"]
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = styles["border"]

    for offset, record in enumerate(frame.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            if isinstance(value, (np.integer,)):
                value = int(value)
            elif isinstance(value, (np.floating,)):
                value = None if np.isnan(value) else float(value)
            cell = worksheet.cell(row=header_row + offset, column=col_idx, value=value)
            cell.font = styles["cell_font"]
            cell.border = styles["border"]
            column = frame.columns[col_idx - 1]
            if column in percent_cols:
                cell.number_format = "0.0000"
            elif column in ("p", "Optimal_p", "Sigma"):
                cell.number_format = "0.00"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit(worksheet, len(frame.columns), header_row, len(frame))
    return header_row


def _write_methodology(worksheet: Worksheet, lines: List[tuple], styles: dict) -> None:
    worksheet.column_dimensions["A"].width = 110
    fonts = {
        "title": styles["title_font"],
        "head":  Font(name=FONT_NAME, bold=True, size=11),
        "note":  styles["note_font"],
        "body":  styles["cell_font"],
        "blank": styles["cell_font"],
    }
    for row, (kind, text) in enumerate(lines, start=1):
        cell = worksheet.cell(row=row, column=1, value=text)
        cell.font = fonts[kind]
        cell.alignment = Alignment(wrap_text=False, vertical="top")


def _replace_sheet(workbook, name: str):
    if name in workbook.sheetnames:
        del workbook[name]
    return workbook.create_sheet(name[:31])


# ==============================================================================
#  MAIN ENTRY POINT
# ==============================================================================

def augment_workbook(filepath: str) -> Path:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Workbook not found: {path.resolve()}")

    frame = load_results(path)
    systems = system_order(frame)
    rules = rule_order(frame)
    sigmas = sigma_order(frame)
    p_values = p_order(frame)

    per_config = aggregate_per_config(frame)

    best_by_eer = build_best_by_eer(per_config, systems, sigmas)
    best_by_rank1 = build_best_by_rank1(per_config, systems, sigmas)
    eer_sensitivity = build_sensitivity(per_config, systems, rules, sigmas, "EER_mean")
    rank1_sensitivity = build_sensitivity(per_config, systems, rules, sigmas, "Rank1_mean")
    p_sensitivity = build_p_sensitivity(per_config, systems, rules, sigmas)
    best_eer_vs_sigma = build_best_eer_vs_sigma(per_config, systems, sigmas)
    eer_vs_sigma_winning = build_eer_vs_sigma_winning_config(per_config, systems, sigmas)

    workbook = load_workbook(path)
    styles = _styles()

    ws = _replace_sheet(workbook, "Methodology")
    _write_methodology(ws, methodology_lines(frame, systems, rules, sigmas, p_values), styles)

    ws = _replace_sheet(workbook, "Best_By_EER")
    _write_table(
        ws, best_by_eer, styles,
        title="Best Configuration per (System, Sigma) \u2014 selected on minimum mean EER",
        subtitle="Rule and p re-optimized independently at every noise level.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%"),
    )

    ws = _replace_sheet(workbook, "Best_by_R1FAR1")
    _write_table(
        ws, best_by_rank1, styles,
        title="Best Configuration per (System, Sigma) \u2014 selected on maximum mean Rank-1 @ FAR=1%",
        subtitle="Independent optimization from Best_By_EER; winning (rule, p) can differ.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%"),
    )

    ws = _replace_sheet(workbook, "EER_Sensitivity_Analysis")
    hr = _write_table(
        ws, eer_sensitivity, styles,
        title="EER Sensitivity Analysis \u2014 mean EER (%) per (System,Sigma) x Rule, averaged over p",
        subtitle="Shows how a rule performs on average at each noise level, independent of p tuning.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(eer_sensitivity.columns))
        ws.conditional_formatting.add(
            f"C{hr+1}:{last_col}{hr+len(eer_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="63BE7B", end_type="max", end_color="F8696B"),
        )

    ws = _replace_sheet(workbook, "R1FAR1_Sensitivity_Analysis")
    hr = _write_table(
        ws, rank1_sensitivity, styles,
        title="Rank-1 @ FAR=1% Sensitivity Analysis \u2014 mean per (System,Sigma) x Rule, averaged over p",
        subtitle="Shows how a rule performs on average at each noise level, independent of p tuning.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(rank1_sensitivity.columns))
        ws.conditional_formatting.add(
            f"C{hr+1}:{last_col}{hr+len(rank1_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="F8696B", end_type="max", end_color="63BE7B"),
        )

    ws = _replace_sheet(workbook, "P_Sensitivity_Analysis")
    _write_table(
        ws, p_sensitivity, styles,
        title="P-Sensitivity Analysis \u2014 EER-optimal p per Rule x System x Sigma (long form)",
        subtitle="One row per (Rule, System, Sigma); filter/sort in Excel to inspect any slice.",
        percent_cols=("EER (%) at Optimal_p",),
    )

    ws = _replace_sheet(workbook, "BestEER_vs_Sigma_AllRules")
    _write_table(
        ws, best_eer_vs_sigma, styles,
        title="Best Achievable EER (%) vs Noise (sigma) \u2014 minimum over all rules & p (5-fold CV mean)",
        subtitle="Per system, the lowest mean EER obtainable at each noise level if the best rule/p for it is used.",
        percent_cols=tuple(systems),
    )

    ws = _replace_sheet(workbook, "EER_vs_Sigma_WinningConfig")
    _write_table(
        ws, eer_vs_sigma_winning, styles,
        title="EER (%) vs Noise (sigma) \u2014 each system's single overall winning (rule,p) held fixed",
        subtitle="Configuration chosen once (best mean EER averaged across ALL sigma levels), then evaluated per sigma.",
        percent_cols=tuple(systems),
    )

    desired_order = [RESULTS_SHEET_NAME, "Methodology", "Best_By_EER", "Best_by_R1FAR1",
                     "EER_Sensitivity_Analysis", "R1FAR1_Sensitivity_Analysis",
                     "P_Sensitivity_Analysis", "BestEER_vs_Sigma_AllRules",
                     "EER_vs_Sigma_WinningConfig"]
    remaining = [s for s in workbook.sheetnames if s not in desired_order]
    workbook._sheets = [workbook[s] for s in desired_order if s in workbook.sheetnames] \
                       + [workbook[s] for s in remaining]

    workbook.save(path)
    return path


if __name__ == "__main__":

    print("Analyse starts  - " * 1)
    filepath = RAW_XLSX_FILE
    print(f"Augmenting workbook: {filepath}")
    output_path = augment_workbook(filepath)
    print("Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, "
          "EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, "
          "P_Sensitivity_Analysis, BestEER_vs_Sigma_AllRules, EER_vs_Sigma_WinningConfig")
    print(f"Saved to: {output_path}")
    print("DONE   " * 3) 

Analyse starts  - 
Augmenting workbook: ./Results/noise_pdep_test_seed62_raw.xlsx
Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, P_Sensitivity_Analysis, BestEER_vs_Sigma_AllRules, EER_vs_Sigma_WinningConfig
Saved to: Results\noise_pdep_test_seed62_raw.xlsx
DONEDONEDONE
